In [38]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))

from sqlalchemy import text
from db.database import engine
from etl.constants import EUROPEAN_COUNTRIES, DECOUPLING_START_YEAR, DECOUPLING_END_YEAR

In [ ]:
# Emissions + urbanization for analysis
query = """
    SELECT
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_per_capita,
        e.co2_total,
        e.co2_per_gdp,
        e.consumption_co2_per_capita,
        e.gdp,
        e.population,
        u.urban_population_pct,
        u.urban_growth_rate,
        u.population_density,
        u.urban_population_total,
        u.pm25_exposure,
        u.electricity_access_pct,
        u.slum_population_pct
    FROM emissions e
    JOIN countries c ON c.id = e.country_id
    LEFT JOIN urbanization u 
        ON u.country_id = e.country_id 
        AND u.year = e.year
    WHERE e.year BETWEEN :start AND :end
    ORDER BY c.iso_code, e.year
"""

with engine.connect() as conn:
    df = pd.read_sql(
        text(query),
        conn,
        params={'start': DECOUPLING_START_YEAR, 'end': DECOUPLING_END_YEAR}
    )

df_eu = df[
    df['iso_code'].isin(EUROPEAN_COUNTRIES) &
    (df['iso_code'] != 'MLT')
].copy()

# GDP per capita
df['gdp_per_capita'] = df['gdp'] / df['population']
df_eu['gdp_per_capita'] = df_eu['gdp'] / df_eu['population']

print(f"Global: {len(df)} rows, {df['iso_code'].nunique()} countries")
print(f"Europe: {len(df_eu)} rows, {df_eu['iso_code'].nunique()} countries")
print(f"\nUrbanization data coverage:")
print(f" urban_population_pct: {df['urban_population_pct'].notna().sum()} rows")
print(f" population_density: {df['population_density'].notna().sum()} rows")
print(f" pm25_exposure: {df['pm25_exposure'].notna().sum()} rows")

2026-06-08 15:57:49,005 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-08 15:57:49,007 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-06-08 15:57:49,011 INFO sqlalchemy.engine.Engine [cached since 4250s ago] {'table_name': <sqlalchemy.sql.elements.TextClause object at 0x00000227EEA80E10>, 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-06-08 15:57:49,013 INFO sqlalchemy.engine.Engine 
    SELECT
        c.name as country,
        c.iso_code,
        e.year,
        e.co2_per_capita,
        e.co2_total,

In [40]:
# Use latest year with good coverage for cross-country comparison
latest = df[
    (df['year'] == 2022) &
    (df['urban_population_pct'].notna()) &
    (df['co2_per_capita'].notna())
].copy()

latest_eu = df_eu[
    (df_eu['year'] == 2022) &
    (df_eu['urban_population_pct'].notna()) &
    (df_eu['co2_per_capita'].notna())
].copy()

print(f"Global snapshot 2022: {len(latest)} countries")
print(f"European snapshot 2022: {len(latest_eu)} countries")
print(f"\nUrbanization range: {latest['urban_population_pct'].min():.1f}% - {latest['urban_population_pct'].max():.1f}%")
print(f"CO2 per capita range: {latest['co2_per_capita'].min():.2f} - {latest['co2_per_capita'].max():.2f} tonnes")

Global snapshot 2022: 204 countries
European snapshot 2022: 33 countries

Urbanization range: 14.7% - 100.0%
CO2 per capita range: 0.06 - 37.89 tonnes


In [41]:
# Global view - color by GDP per capita (wealth as third variable)
fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='co2_per_capita',
    color='gdp_per_capita',
    size='population',
    size_max=75,
    hover_name='country',
    color_continuous_scale='RdYlGn',
    title='Urbanization vs CO2 per Capita - Global (2022)<br>'
          '<sup>Color = GDP per capita | Size = population</sup>',
    labels={
        'urban_population_pct': 'Urban population (%)',
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'gdp_per_capita': 'GDP per capita'
    },
    height=1000
)

# Add trend line
fig.update_traces(marker=dict(opacity=0.9))
fig.show()

# Correlation
corr = latest[['urban_population_pct', 'co2_per_capita', 'gdp_per_capita']].corr()
print("\nCorrelation matrix:")
print(corr.round(3))


Correlation matrix:
                      urban_population_pct  co2_per_capita  gdp_per_capita
urban_population_pct                 1.000           0.476           0.617
co2_per_capita                       0.476           1.000           0.742
gdp_per_capita                       0.617           0.742           1.000


In [ ]:
# Global view - but with consumption co2
fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='consumption_co2_per_capita',
    color='gdp_per_capita',
    size='population',
    size_max=75,
    hover_name='country',
    color_continuous_scale='RdYlGn',
    title='Urbanization vs Consumption CO2 per Capita - Global (2022)<br>'
          '<sup>Color = GDP per capita | Size = population</sup>',
    labels={
        'urban_population_pct': 'Urban population (%)',
        'consumption_co2_per_capita': 'Consumption CO2 per capita (tonnes)',
        'gdp_per_capita': 'GDP per capita'
    },
    height=1000
)

fig.update_traces(marker=dict(opacity=0.9))
fig.show()

# Correlation
corr = latest[['urban_population_pct', 'consumption_co2_per_capita', 'gdp_per_capita']].corr()
print("\nCorrelation matrix:")
print(corr.round(3))


Correlation matrix:
                            urban_population_pct  consumption_co2_per_capita  \
urban_population_pct                       1.000                       0.634   
consumption_co2_per_capita                 0.634                       1.000   
gdp_per_capita                             0.617                       0.801   

                            gdp_per_capita  
urban_population_pct                 0.617  
consumption_co2_per_capita           0.801  
gdp_per_capita                       1.000  


In [139]:
# Finding urbanization-CO2 link

# Split into income groups by GDP per capita
latest['income_group'] = pd.qcut(
    latest['gdp_per_capita'],
    q=5,
    labels=['Low income', 'Lower-middle', 'Middle', 'Upper-middle', 'High income']
)
latest = latest.dropna(subset=['income_group'])
fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='co2_per_capita',
    color='income_group',
    hover_name='country',
    facet_col='income_group',
    trendline='ols',
    title='Urbanization vs CO2 per Capita - by Income Group<br>'
          '<sup>Does the relationship hold within income groups?</sup>',
    labels={
        'urban_population_pct': 'Urban pop. (%)',
        'co2_per_capita': 'CO2 per capita (tonnes)'
    },
    height=500
)
fig.show()

# Second chart with consumption co2
latest['income_group'] = pd.qcut(
    latest['gdp_per_capita'],
    q=5,
    labels=['Low income', 'Lower-middle', 'Middle', 'Upper-middle', 'High income']
)
latest = latest.dropna(subset=['income_group'])
fig = px.scatter(
    latest,
    x='urban_population_pct',
    y='consumption_co2_per_capita',
    color='income_group',
    hover_name='country',
    facet_col='income_group',
    trendline='ols',
    title='Urbanization vs Consumption CO2 per Capita - by Income Group<br>',
    labels={
        'urban_population_pct': 'Urban pop. (%)',
        'consumption_co2_per_capita': 'CO2 per capita (tonnes)'
    },
    height=500
)
fig.show()

# Correlation within each income group
print("Correlation (urban_pct vs co2_per_capita) by income group:")
for group in latest['income_group'].cat.categories:
    subset = latest[latest['income_group'] == group]
    corr = subset['urban_population_pct'].corr(subset['co2_per_capita'])
    print(f"  {group}: {corr:.3f} (n={len(subset)})")

# Correlation within each income group
print("\nCorrelation (urban_pct vs consumption_co2_per_capita) by income group:")
for group in latest['income_group'].cat.categories:
    subset = latest[latest['income_group'] == group]
    corr = subset['urban_population_pct'].corr(subset['consumption_co2_per_capita'])
    print(f"  {group}: {corr:.3f} (n={len(subset)})")

Correlation (urban_pct vs co2_per_capita) by income group:
  Low income: 0.530 (n=33)
  Lower-middle: -0.007 (n=32)
  Middle: -0.065 (n=33)
  Upper-middle: -0.143 (n=32)
  High income: 0.337 (n=33)

Correlation (urban_pct vs consumption_co2_per_capita) by income group:
  Low income: 0.835 (n=33)
  Lower-middle: 0.131 (n=32)
  Middle: 0.074 (n=33)
  Upper-middle: -0.257 (n=32)
  High income: 0.559 (n=33)


In [87]:
# Population density vs CO2 per capita
# Theory: denser cities = more efficient = less emissions per person

density_df = latest[latest['population_density'].notna()].copy()

fig = px.scatter(
    density_df,
    x='population_density',
    y='co2_per_capita',
    color='urban_population_pct',
    hover_name='country',
    size='population',
    size_max=75,
    color_continuous_scale='Reds',
    title='Population Density vs CO2 per Capita (2022)<br>'
          '<sup>Do denser countries emit less per person?</sup>',
    labels={
        'population_density': 'Population density (people/km²)',
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'urban_population_pct': 'Urban<br>population (%)',
    },
    height=700,
    log_x=True  # log scale - density varies hugely
)
fig.show()

# European view - less noise, more comparable countries
fig_eu = px.scatter(
    latest_eu[latest_eu['population_density'].notna()],
    x='population_density',
    y='co2_per_capita',
    color='urban_population_pct',
    text='country',
    color_continuous_scale='reds',
    title='Density vs CO2 - Europe only (2022)',
    labels={
        'population_density': 'Population density (people/km²)',
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'urban_population_pct': 'Urban<br>population (%)'
    },
    height=700
)

custom_positions = {
    "Croatia": "bottom center",
    "Sweden": "bottom center",
    "Romania": "middle left",
    "Lithuania": "middle left",
    "Greece": "middle left",
    "Spain": "middle left",
    "Netherlands": "top left",
}
positions = [
    custom_positions.get(country, "middle right")
    for country in latest_eu["country"]
]
fig_eu.update_traces(textposition=positions)
fig_eu.show()

corr_density = density_df['population_density'].corr(density_df['co2_per_capita'])
print(f"\nGlobal correlation (density vs co2_per_capita): {corr_density:.3f}")
corr_eu_density = latest_eu['population_density'].corr(latest_eu['co2_per_capita'])
print(f"Europe correlation (density vs co2_per_capita): {corr_eu_density:.3f}")


Global correlation (density vs co2_per_capita): 0.005
Europe correlation (density vs co2_per_capita): 0.283


In [ ]:
# PM2.5 = air pollution proxy
# If PM2.5 drops as CO2 drops - same root cause (fossil fuels)
# If they diverge - different drivers
latest = df[
    (df['year'] == 2020) &
    (df['urban_population_pct'].notna()) &
    (df['co2_per_capita'].notna())
].copy()

pm25_df = latest[latest['pm25_exposure'].notna()].copy()

fig = px.scatter(
    pm25_df,
    x='co2_per_capita',
    y='pm25_exposure',
    color='urban_population_pct',
    hover_name='country',
    size='population',
    size_max=75,
    color_continuous_scale='RdYlGn_r',
    trendline='ols',
    title='CO2 per Capita vs PM2.5 Exposure (World, 2020)<br>'
          '<sup>Do air pollution and CO2 emissions share the same drivers?</sup>',
    labels={
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'pm25_exposure': 'PM2.5 exposure (μg/m³)',
        'urban_population_pct': 'Urban<br> population %'
    },
    height=700
)
fig.add_hline(
    y=5,
    line_dash='dash',
    line_color='red',
    line_width=1,
    annotation_text='WHO guideline (5 μg/m³)'
)
fig.add_hline(
    y=15,
    line_dash='dash',
    line_color='red',
    line_width=1,
    annotation_text='WHO interim target (15 μg/m³)'
)
fig.update_traces(marker=dict(opacity=0.8))
fig.show()


pm25_eu_time = df_eu[df_eu['pm25_exposure'].notna()
                ].groupby('year')[['co2_per_capita', 'pm25_exposure']
                ].mean().reset_index()

fig2 = make_subplots(specs=[[{"secondary_y": True}]])
fig2.add_trace(
    go.Scatter(x=pm25_eu_time['year'], y=pm25_eu_time['co2_per_capita'],
               name='CO2 per capita (avg)', line=dict(color='red')),
    secondary_y=False
)
fig2.add_trace(
    go.Scatter(x=pm25_eu_time['year'], y=pm25_eu_time['pm25_exposure'],
               name='PM2.5 exposure (avg)', line=dict(color='brown', dash='dash')),
    secondary_y=True
)
fig2.update_layout(
    title='Europe average: CO2 per capita vs PM2.5 over time<br>'
          '<sup>Do they improve together?</sup>',
    height=500
)
fig2.show()


corr_pm25 = pm25_df['co2_per_capita'].corr(pm25_df['pm25_exposure'])
print(f"\nCorrelation (co2_per_capita vs pm25): {corr_pm25:.3f}")


Correlation (co2_per_capita vs pm25): -0.024

European correlation (urban_population_pct vs pm25): -0.534


In [ ]:
#PM2.5 vs CO2 in Europe
latest_eu = df_eu[
    (df_eu['year'] == 2020) &
    (df_eu['urban_population_pct'].notna()) &
    (df_eu['co2_per_capita'].notna())
].copy()

pm25_eu_df = latest_eu[latest_eu['pm25_exposure'].notna()].copy()

fig_europe = px.scatter(
    pm25_eu_df,
    x='co2_per_capita',
    y='pm25_exposure',
    color='urban_population_pct',
    hover_name='country',
    size='population',
    size_max=55,
    color_continuous_scale='RdYlGn_r',
    trendline='ols',
    title='CO2 per Capita vs PM2.5 Exposure (Europe, 2020)<br>',
    labels={
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'pm25_exposure': 'PM2.5 exposure (μg/m³)',
        'urban_population_pct': 'Urban<br>population %'
    },
    height=700
)

# WHO guideline
fig_europe.add_hline(
    y=5,
    line_dash='dash',
    line_color='red',
    line_width=1,
    annotation_text='WHO guideline (5 μg/m³)'
)
fig_europe.show()


#PM2.5 vs consumption in Europe
latest_eu_cons = df_eu[
    (df_eu['year'] == 2020) &
    (df_eu['urban_population_pct'].notna()) &
    (df_eu['consumption_co2_per_capita'].notna())
].copy()

pm25_eu__cons_df = latest_eu_cons[latest_eu_cons['pm25_exposure'].notna()].copy()

fig_europe = px.scatter(
    pm25_eu__cons_df,
    x='consumption_co2_per_capita',
    y='pm25_exposure',
    color='urban_population_pct',
    hover_name='country',
    size='population',
    size_max=55,
    color_continuous_scale='RdYlGn_r',
    trendline='ols',
    title='Consumption CO2 per Capita vs PM2.5 Exposure (Europe, 2020)<br>',
    labels={
        'consumption_co2_per_capita': 'Consumption CO2 per capita (tonnes)',
        'pm25_exposure': 'PM2.5 exposure (μg/m³)',
        'urban_population_pct': 'Urban<br>population %'
    },
    height=700
)
fig_europe.show()

corr_pm25 = pm25_eu_df['co2_per_capita'].corr(pm25_eu_df['pm25_exposure'])
print(f"European correlation (co2_per_capita vs pm25): {corr_pm25:.3f}")
corr_pm25_cons = pm25_eu__cons_df['consumption_co2_per_capita'].corr(pm25_eu__cons_df['pm25_exposure'])
print(f"\nEuropean correlation (consumption_co2_per_capita vs pm25): {corr_pm25_cons:.3f}")
corr_pm25_urb = pm25_eu_df['urban_population_pct'].corr(pm25_eu_df['pm25_exposure'])
print(f"\nEuropean correlation (urban_population_pct vs pm25): {corr_pm25_urb:.3f}")

European correlation (co2_per_capita vs pm25): -0.225

European correlation (consumption_co2_per_capita vs pm25): -0.410

European correlation (urban_population_pct vs pm25): -0.534


In [ ]:
# Fast urbanizing countries - do emissions go up or down?
# This is the key development question

# Calculate emissions change 2000-2022 per country
def calc_change(group, col, year_start, year_end):
    start = group[group['year'] == year_start][col].values
    end = group[group['year'] == year_end][col].values
    if len(start) == 0 or len(end) == 0:
        return np.nan
    if pd.isna(start[0]) or pd.isna(end[0]) or start[0] == 0:
        return np.nan
    return (end[0] - start[0]) / start[0] * 100

changes = df.groupby('iso_code').apply(lambda g: pd.Series({
    'country': g['country'].iloc[0],
    'co2_change_pct': calc_change(g, 'co2_per_capita', 2000, 2022),
    'urban_growth_avg': g[g['year'].between(2000, 2022)]['urban_growth_rate'].mean(),
    'gdp_per_capita_2022': g[g['year'] == 2022]['gdp_per_capita'].values[0]
        if len(g[g['year'] == 2022]) > 0 else np.nan
})).reset_index().dropna()

# Color by income
changes['income_group'] = pd.qcut(
    changes['gdp_per_capita_2022'],
    q=4,
    labels=['Low', 'Lower-mid', 'Upper-mid', 'High']
)

fig = px.scatter(
    changes,
    x='urban_growth_avg',
    y='co2_change_pct',
    color='income_group',
    hover_name='country',
    trendline='ols',
    title='Urban Growth Rate vs CO2 Change (2000-2022)<br>'
          '<sup>Does faster urbanization drive more emissions?</sup>',
    labels={
        'urban_growth_avg': 'Avg annual urban growth rate (%)',
        'co2_change_pct': 'CO2 per capita change 2000-2022 (%)'
    },
    height=600
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.show()

In [ ]:
# Which European countries combine:
# High urbanization (>70%)
# Low CO2 per capita (<5 tonnes)

latest_eu_full = latest_eu[
    latest_eu['population_density'].notna() &
    latest_eu['urban_population_pct'].notna()
].copy()

latest_eu_full['gdp_per_capita'] = latest_eu_full['gdp'] / latest_eu_full['population']

fig = px.scatter(
    latest_eu_full,
    x='urban_population_pct',
    y='co2_per_capita',
    size='gdp_per_capita',
    color='population_density',
    text='iso_code',
    color_continuous_scale='Blues',
    title='European Cities Sweet Spot: high urbanization and low emissions (2022)<br>'
          '<sup>Size = GDP per capita | Color = population density</sup>',
    labels={
        'urban_population_pct': 'Urban population (%)',
        'co2_per_capita': 'CO2 per capita (tonnes)',
        'population_density': 'Density (people/km²)'
    },
    height=600
)

# Sweet spot lines
fig.add_hline(y=6, line_dash='dash', line_color='green',
              annotation_text='6t CO2 target')
fig.add_vline(x=70, line_dash='dash', line_color='blue',
              annotation_text='70% urban')

fig.update_traces(textposition='top center')
fig.show()

#Consumption
fig2 = px.scatter(
    latest_eu_full,
    x='urban_population_pct',
    y='consumption_co2_per_capita',
    size='gdp_per_capita',
    color='population_density',
    text='iso_code',
    color_continuous_scale='Blues',
    title='European Cities Sweet Spot: High Urbanization + Low Emissions (2022)<br>'
          '<sup>Size = GDP per capita | Color = population density</sup>',
    labels={
        'urban_population_pct': 'Urban population (%)',
        'consumption_co2_per_capita': 'Consumption CO2 per capita (tonnes)',
        'population_density': 'Density (people/km²)'
    },
    height=600
)

# Sweet spot lines
fig2.add_hline(y=6, line_dash='dash', line_color='green',
              annotation_text='6t CO2 target')
fig2.add_vline(x=70, line_dash='dash', line_color='blue',
              annotation_text='70% urban')

fig2.update_traces(textposition='top center')
fig2.show()

# Print the sweet spot countries
sweet_spot = latest_eu_full[
    (latest_eu_full['urban_population_pct'] > 70) &
    (latest_eu_full['co2_per_capita'] < 6)
][['country', 'urban_population_pct', 'co2_per_capita',
   'population_density', 'gdp_per_capita']].sort_values('co2_per_capita')

print("Sweet spot countries (>70% urban, <6t CO2 per capita):")
print(sweet_spot.to_string(index=False))

# Print the sweet spot countries (consumption co2)
sweet_spot_consumption = latest_eu_full[
    (latest_eu_full['urban_population_pct'] > 70) &
    (latest_eu_full['consumption_co2_per_capita'] < 6)
][['country', 'urban_population_pct', 'consumption_co2_per_capita',
   'population_density', 'gdp_per_capita']].sort_values('consumption_co2_per_capita')

print("Sweet spot countries (>70% urban, <6t Consumption CO2 per capita):")
print(sweet_spot_consumption.to_string(index=False))

Sweet spot countries (>70% urban, <6t CO2 per capita):
       country  urban_population_pct  co2_per_capita  population_density  gdp_per_capita
        Sweden             87.942405           3.537           25.420944    44127.810225
   Switzerland             82.341684           3.975          218.634470    60863.549407
        France             78.711913           4.272          125.431135    37001.502645
         Spain             79.695567           4.441           94.802911    30631.791174
       Hungary             70.291733           4.832          105.965582    26110.564898
United Kingdom             82.900002           4.844          275.881453    34665.769583
       Denmark             88.239449           4.853          145.785100    47167.454620
        Greece             78.306684           5.194           82.999216    21475.827222
      Bulgaria             73.323990           5.274           60.341710    17940.433295
Sweet spot countries (>70% urban, <6t Consumption CO2 p

In [ ]:
print("=" * 30)
print("NOTEBOOK 05 - KEY FINDINGS")
print("=" * 30)

print("""
Finding 1: Urbanization correlates with CO2 - but GDP is the real driver
   Global correlations (2022):
    Urbanization - territorial CO2:   r = 0.476
    Urbanization - consumption CO2:   r = 0.634
    GDP per capita - CO2:             r = 0.742  - strongest

   The urbanization-CO2 link is mostly a wealth proxy.
   The jump to r = 0.634 with consumption CO2 is the honest signal -
   urbanized lifestyles consume more, but that's income, not cities.

      

      Correlation (urban_pct vs co2_per_capita) by income group:
      Low income: 0.530 (n=33)
      Lower-middle: -0.007 (n=32)
      Middle: -0.065 (n=33)
      Upper-middle: -0.143 (n=32)
      High income: 0.337 (n=33)

      Correlation (urban_pct vs consumption_co2_per_capita) by income group:
      Low income: 0.835 (n=33)
      Lower-middle: 0.131 (n=32)
      Middle: 0.074 (n=33)
      Upper-middle: -0.257 (n=32)
      High income: 0.559 (n=33)
Finding 2: Within income groups, the link disappears or reverses
   Urbanization vs CO2 by income group:
      Low income: 0.530
      Lower-middle: -0.007
      Middle: -0.065
      Upper-middle: -0.143
      High income: 0.337

   Ubranization vs Consumption CO2 by income group:
      Low income: 0.835 - far more stronger correlation
      Lower-middle: 0.131
      Middle: 0.074
      Upper-middle: -0.257 - do their best with decoupling
      High income: 0.559 - consumption domination

   Urban efficiency is real, but only visible once income is controlled.
   At high income, consumption lifestyles dominate again.

Finding 3: Population density has no global relationship with CO2
   Global r = 0.005 - effectively zero.
   Europe only: r = 0.283 - modest, due to homogeneous sample.
   Energy mix matters far more than physical density.

Finding 4: In Europe, more urbanized = better air quality (PM2.5)
   Urbanization - PM2.5 (Europe):    r = -0.534
   CO2 - PM2.5 (global):             r = -0.024  - no relationship

   CO2 and air pollution are globally decoupled (different sources).
   In Europe, the link reflects cleaner energy and stronger policy -
   not an intrinsic property of cities.
   Worst air quality is typically biomass burning in lower-income countries,
   not fossil fuel heavyweights.

Finding 5: European "sweet spot" - high urbanization + low emissions
   Above 70% urban AND below 6t territorial CO2/capita (2022):
    Sweden, Switzerland, France, Spain, Hungary, UK, Denmark, Greece, Bulgaria

   Consumption CO2 stress test (<6t):
    Bulgaria, Spain, Sweden only — most others still offshore emissions.
    Switzerland most exposed: territorial at 75 (index), consumption at 145.

Finding 6: Urbanization is a red herring
   What predicts low emissions is energy mix and consumption level,
   not urban structure. Sweden's success is hydro/nuclear + policy.
   Cyprus is 67% urban and still growing emissions with GDP.

   Next chapter (transition leaders) should focus on:
   co2_per_unit_energy and renewables_share_energy as primary variables.
""")

NOTEBOOK 05 - KEY FINDINGS

Finding 1: Urbanization correlates with CO2 - but GDP is the real driver
   Global correlations (2022):
    Urbanization - territorial CO2:   r = 0.476
    Urbanization - consumption CO2:   r = 0.634
    GDP per capita - CO2:             r = 0.742  - strongest

   The urbanization-CO2 link is mostly a wealth proxy.
   The jump to r = 0.634 with consumption CO2 is the honest signal -
   urbanized lifestyles consume more, but that's income, not cities.

Finding 2: Within income groups, the link disappears or reverses
   Urbanization vs CO2 by income quintile:
    Low income:    +0.557
    Lower-middle:  +0.221
    Middle:        -0.072
    Upper-middle:  -0.141  - more urban = less CO2
    High income:   +0.342

   Urban efficiency is real, but only visible once income is controlled.
   At high income, consumption lifestyles dominate again.

Finding 3: Population density has no global relationship with CO2
   Global r = 0.005 - effectively zero.
   Europe only